In [52]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [64]:
def combine_all_years_to_df(data_dir, years=range(2016, 2024)):
    '''
    Function to combine data for multiple years into a single DataFrame.
    
    Args: 
        - years (list of int): List of years to combine.
    Returns:
        - combined_df (pd.DataFrame): A DataFrame containing all combined data for the given years.
    '''
    combined = []
    for year in years:
        path = os.path.join(data_dir, f"playbyplay-{str(year)}.json")
        df = pd.read_json(path)
        combined.append(df)
    
    combined_df = pd.concat(combined, ignore_index=True)
    return combined_df

def normalize_game_plays(game_row: pd.Series) -> pd.DataFrame:

    plays = game_row.get("plays", [])
    
    if not isinstance(plays, list) or len(plays) == 0:
        return pd.DataFrame()

    return pd.json_normalize(plays)


def infer_attacking_direction(plays: pd.DataFrame) -> pd.Series:

    tmp = plays.dropna(subset=["details.xCoord"]).copy()
    if tmp.empty:
        return pd.Series([None] * len(plays), index=plays.index)

    tmp["teamId"] = tmp["details.eventOwnerTeamId"]
    tmp["period"] = tmp["periodDescriptor.number"]
    mean_x = tmp.groupby(["teamId", "period"])["details.xCoord"].mean()

    cache = {}
    for (team_id, period), mx in mean_x.items():
        cache.setdefault(period, {})[team_id] = "left" if mx < 0 else "right"

    return plays.apply(lambda r: cache.get(r.get("periodDescriptor.number"), {}).get(r.get("details.eventOwnerTeamId")),
                       axis=1)


def compute_empty_net(plays: pd.DataFrame, home_team_id: int, away_team_id: int) -> pd.Series:

    if "situationCode" not in plays.columns:
        return pd.Series(0, index=plays.index, dtype=int)

    situation = plays["situationCode"].astype(str)
    away_goalie_in = situation.str[0] == "1"
    home_goalie_in = situation.str[3] == "1"

    shooter_team = plays["details.eventOwnerTeamId"]
    result = pd.Series(0, index=plays.index, dtype=int)
    home_shooting = shooter_team == home_team_id
    away_shooting = shooter_team == away_team_id

    result[home_shooting] = (~away_goalie_in[home_shooting]).astype(int)
    result[away_shooting] = (~home_goalie_in[away_shooting]).astype(int)
    return result


def calculate_net_properties(row):

    left_net_x = -89.0
    right_net_x = 89.0

    x = row.get("x")
    y = row.get("y")
    direction = row.get("attackingDirection")

    if pd.isna(x) or pd.isna(y):
        return np.nan, np.nan

    if direction == "right":
        net_x = right_net_x
    elif direction == "left":
        net_x = left_net_x
    else:
        net_x = right_net_x if x >= 0 else left_net_x

    net_y = 0
    dx =  abs(net_x - x)
    dy = net_y - y
    distance = np.sqrt(dx**2 + dy**2)

    angle_rad = np.arctan2(dy, dx)
    angle_deg = np.degrees(angle_rad)
    
    return distance, angle_deg


def extract_shot_features(df: pd.DataFrame) -> pd.DataFrame:

    all_plays = []

    for _, game in df.iterrows():
        plays = normalize_game_plays(game)
        if plays.empty:
            continue
        
        
        plays["x"] = pd.to_numeric(plays["details.xCoord"], errors="coerce")
        plays["y"] = pd.to_numeric(plays["details.yCoord"], errors="coerce")

        plays["attackingDirection"] = infer_attacking_direction(plays)

        game_id = game["id"]
        home_id = game["homeTeam"]["id"]
        away_id = game["awayTeam"]["id"]
        plays["empty_net"] = compute_empty_net(plays, home_id, away_id)

        plays[["distance_from_net_ft", "shot_angle_deg"]] = plays.apply(
            calculate_net_properties, axis=1, result_type="expand")
        
        plays['distance_from_net_ft'] = plays['distance_from_net_ft'].round(2)
        plays['shot_angle_deg'] = plays['shot_angle_deg'].round(2)
        plays["is_goal"] = (plays["typeDescKey"] == "goal").astype(int)

        plays["game_id"] = game_id
        plays["home_id"] = home_id
        plays["away_id"] = away_id
        
        all_plays.append(plays)

    if not all_plays:
        return pd.DataFrame(columns=plays.columns)

    final = pd.concat(all_plays, ignore_index=True)
    return final

def mmss_to_seconds(mmss):
    minutes, seconds = map(int, mmss.split(':'))
    return (minutes * 60) + seconds

def calculate_prev_net_properties(row):

    left_net_x = -89.0
    right_net_x = 89.0

    x = row.get("prev_x")
    y = row.get("prev_y")
    direction = row.get("attackingDirection")

    if pd.isna(x) or pd.isna(y):
        return np.nan, np.nan

    if direction == "right":
        net_x = right_net_x
    elif direction == "left":
        net_x = left_net_x
    else:
        net_x = right_net_x if x >= 0 else left_net_x

    net_y = 0
    dx = abs(net_x - x)
    dy = net_y - y
    distance = np.sqrt(dx**2 + dy**2)

    angle_rad = np.arctan2(dy, dx)
    angle_deg = np.degrees(angle_rad)
    return distance, angle_deg

In [65]:
df = combine_all_years_to_df(data_dir='data', years=range(2016,2021))
df.tail()

,id,season,gameType,limitedScoring,gameDate,venue,venueLocation,startTimeUTC,easternUTCOffset,venueUTCOffset,...,otInUse,clock,displayPeriod,maxPeriods,gameOutcome,plays,rosterSpots,regPeriods,summary,specialEvent
6177,2020030411,20202021,3,False,2021-06-28,{'default': 'Amalie Arena'},{'default': 'Tampa'},2021-06-29T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 14, 'playerId': 8470147, 'firstNam...",3,{},NaN
6178,2020030412,20202021,3,False,2021-06-30,{'default': 'Amalie Arena'},{'default': 'Tampa'},2021-07-01T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 14, 'playerId': 8470147, 'firstNam...",3,{},NaN
6179,2020030413,20202021,3,False,2021-07-02,{'default': 'Centre Bell'},"{'default': 'Montreal', 'fr': 'Montréal'}",2021-07-03T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 14, 'playerId': 8470147, 'firstNam...",3,{},NaN
6180,2020030414,20202021,3,False,2021-07-05,{'default': 'Centre Bell'},"{'default': 'Montreal', 'fr': 'Montréal'}",2021-07-06T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '16:03', 'secondsRemaining':...",1,NaN,"{'lastPeriodType': 'OT', 'otPeriods': 1}","[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 14, 'playerId': 8470147, 'firstNam...",3,{},NaN
6181,2020030415,20202021,3,False,2021-07-07,{'default': 'Amalie Arena'},{'default': 'Tampa'},2021-07-08T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 14, 'playerId': 8470147, 'firstNam...",3,{},NaN


In [69]:
df2 = df[df['gameType'] == 2]
df3 = df[df['gameType'] == 3]
test_df = df.copy()
print(df2.columns)
print(df3.columns)
cols_A = set(df2.columns)
cols_B = set(df3.columns)

# 3. Find the symmetric difference
column_difference = cols_A.symmetric_difference(cols_B)

print("Columns that are not in both DataFrames:")
print(list(column_difference))
# test_df.to_csv("test_playoff_df.csv")

Index(['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue',
       'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset',
       'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor',
       'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock',
       'displayPeriod', 'maxPeriods', 'gameOutcome', 'plays', 'rosterSpots',
       'regPeriods', 'summary', 'specialEvent'],
      dtype='object')
Index(['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue',
       'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset',
       'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor',
       'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock',
       'displayPeriod', 'maxPeriods', 'gameOutcome', 'plays', 'rosterSpots',
       'regPeriods', 'summary', 'specialEvent'],
      dtype='object')
Columns that are not in both DataFrames:
[]


In [56]:
SHOT_EVENTS = ['shot-on-goal', 'blocked-shot', 'missed-shot']

test_processed_df = extract_shot_features(test_df)
test_processed_df['game_seconds'] = test_processed_df['timeInPeriod'].apply(mmss_to_seconds)
test_processed_df['prev_event'] = test_processed_df['typeDescKey'].shift(1)
test_processed_df['prev_x'] = test_processed_df['details.xCoord'].shift(1)
test_processed_df['prev_y'] = test_processed_df['details.yCoord'].shift(1)
test_processed_df['prev_shot_angle'] = test_processed_df['shot_angle_deg'].shift(1)
test_processed_df['seconds_since_prev'] = test_processed_df['game_seconds'] - test_processed_df['game_seconds'].shift(1)
test_processed_df['distance_to_prev'] = (test_processed_df['distance_from_net_ft'] - test_processed_df['distance_from_net_ft'].shift(1)).abs().round(2)
test_processed_df['rebound'] = test_processed_df['prev_event'].isin(SHOT_EVENTS)

test_processed_df[["prev_distance_from_net", "prev_shot_angle"]] = test_processed_df.apply(calculate_prev_net_properties, 
                                                                                             axis=1, 
                                                                                             result_type="expand").round(2)

test_processed_df['shot_angle_change'] = np.where(test_processed_df['rebound'],
                                                   np.abs(test_processed_df['shot_angle_deg'] - test_processed_df['prev_shot_angle']),
                                                   np.nan).round(2)                                                   

test_processed_df['speed_from_prev'] = (test_processed_df['distance_to_prev'] / test_processed_df['seconds_since_prev']).round(2)


In [57]:
test_processed_df.to_csv("4_test_playoff_engineered_df.csv", index=False)


In [58]:
EVENT_TYPE = {"shot-on-goal", "goal", "blocked-shot", "missed-shot"}

test_processed_df = test_processed_df[test_processed_df['typeDescKey'].isin(EVENT_TYPE)] 
test_processed_df = test_processed_df.dropna(subset=['distance_from_net_ft', 'shot_angle_deg'])

test_processed_df = test_processed_df[['is_goal', 'game_seconds', 'periodDescriptor.number','x','y', 'distance_from_net_ft', 'shot_angle_deg','details.shotType', 'prev_event', 'prev_x', 'prev_y', 'seconds_since_prev', 'distance_to_prev', 'rebound', 'shot_angle_change', 'speed_from_prev']].copy()
test_processed_df['rebound'] = test_processed_df['rebound'].astype(int)
test_processed_df['speed_from_prev'] = test_processed_df['speed_from_prev'].replace([np.inf, -np.inf], 0).fillna(0)

categorical_cols = ['periodDescriptor.number', 'details.shotType', 'prev_event']

test_processed_df = pd.get_dummies(test_processed_df, 
                                   columns=categorical_cols, 
                                   prefix=categorical_cols,
                                   drop_first=False,
                                   dtype=int)


In [59]:
test_processed_df.to_csv("5_test_playoff_engineered_df.csv", index=False)

In [60]:
# Coordinates for determining the "Slot"
X_FAR_LIMIT = 59 
X_NEAR_LIMIT = 89 
Y_LIMIT = 4      

# Plays of interest
EVENT_TYPES = {'goal', 'shot-on-goal', 'blocked-shot', 'missed-shot'}

def time_to_seconds(time_str):
    """
    Converts a time format to total seconds.
    Args:
        - time_str (str): The time in MM:SS for the Period time in a hockey game.
    """
    if pd.isna(time_str):
        return np.nan
    try:
        m, s = map(float, time_str.split(':'))
        return m * 60 + s
    except ValueError:
        return np.nan

In [61]:
test_engineered_df = pd.read_csv("4_test_playoff_engineered_df.csv", low_memory=False)

rename_dict = {
    'timeInPeriod': 'time_in_period',
    'timeRemaining': 'time_remaining_period',
    'periodDescriptor.number': 'period_number',
    'periodDescriptor.periodType': 'period_type',
    'periodDescriptor.maxRegulationPeriods': 'max_reg_periods',
    'game_seconds': 'game_time_sec',
    'typeCode': 'event_type_code',
    'typeDescKey': 'event_type_desc',
    'situationCode': 'game_situation_code',
    'details.zoneCode': 'zone_code',
    'x': 'shot_x',
    'y': 'shot_y',
    'distance_from_net_ft': 'shot_distance',
    'shot_angle_deg': 'shot_angle',
    'details.eventOwnerTeamId': 'owner_team_id',
    'details.losingPlayerId': 'losing_player_id',
    'details.winningPlayerId': 'winning_player_id',
    'details.playerId': 'player_id',
    'details.blockingPlayerId': 'blocker_id',
    'details.shootingPlayerId': 'shooter_id',
    'details.goalieInNetId': 'goalie_id',
    'details.hittingPlayerId': 'hitter_id',
    'details.hitteePlayerId': 'hittee_id',
    'details.scoringPlayerId': 'scorer_id',
    'details.assist1PlayerId': 'assist1_id',
    'details.assist2PlayerId': 'assist2_id',
    'details.committedByPlayerId': 'committed_by_id',
    'details.drawnByPlayerId': 'drawn_by_id',
    'details.servedByPlayerId': 'served_by_id',
    'details.shotType': 'shot_type',
    'details.awaySOG': 'away_sog',
    'details.homeSOG': 'home_sog',
    'details.awayScore': 'score_away',
    'details.homeScore': 'score_home',
    'prev_event': 'prev_event_type',
    'seconds_since_prev': 'time_since_prev_event',
    'distance_to_prev': 'dist_from_prev_event',
    'prev_distance_from_net': 'prev_shot_distance',
    'rebound': 'is_rebound',
    'shot_angle_change': 'angle_change',
    'speed_from_prev': 'speed'
}
test_engineered_df.rename(columns=rename_dict, inplace=True)

columns_to_drop = [
    'eventId', 'event_type_code', 'sortOrder', 'period_type',
    'details.xCoord', 'details.yCoord', 'details.reason',
    'details.secondaryReason', 'details.discreteClip',
    'homeTeamDefendingSide', 'details.descKey', 'details.duration',
    'prev_x', 'prev_y', 'prev_shot_angle',
    'details.scoringPlayerTotal', 'details.assist1PlayerTotal', 
    'details.assist2PlayerTotal'
]
test_engineered_df.drop(columns=columns_to_drop, errors='ignore', inplace=True)


print("Performing additional engineering features to add to the features engineered in Part 4...")

test_engineered_df['situation_code_str'] = test_engineered_df['game_situation_code'].astype(str).str.zfill(4)
test_engineered_df['away_goalie_on_ice'] = test_engineered_df['situation_code_str'].str[0] == "1"
test_engineered_df['home_goalie_on_ice'] = test_engineered_df['situation_code_str'].str[3] == "1"

test_engineered_df = test_engineered_df.assign(away_skaters=pd.to_numeric(test_engineered_df['situation_code_str'].str[1], errors='coerce').fillna(5).astype(int),
               home_skaters=pd.to_numeric(test_engineered_df['situation_code_str'].str[2], errors='coerce').fillna(5).astype(int))

test_engineered_df['is_empty_net'] = np.where(test_engineered_df['owner_team_id'] == test_engineered_df['home_id'],
                                  ~test_engineered_df['away_goalie_on_ice'],
                                  ~test_engineered_df['home_goalie_on_ice']).astype(int)

test_engineered_df['shooter_skater_diff'] = np.where(test_engineered_df['owner_team_id'] == test_engineered_df['home_id'],
                                     test_engineered_df['home_skaters'] - test_engineered_df['away_skaters'],
                                     test_engineered_df['away_skaters'] - test_engineered_df['home_skaters'])

test_engineered_df['is_even_strength'] = (test_engineered_df['shooter_skater_diff'] == 0).astype(int)
test_engineered_df['is_power_play']    = (test_engineered_df['shooter_skater_diff'] > 0).astype(int)
test_engineered_df['is_shorthanded']   = (test_engineered_df['shooter_skater_diff'] < 0).astype(int)

test_engineered_df.drop(columns=['situation_code_str', 'away_goalie_on_ice', 'home_goalie_on_ice',
                 'away_skaters', 'home_skaters', 'shooter_skater_diff'], inplace=True)

cols_to_fill = ['away_sog', 'home_sog', 'score_away', 'score_home']
test_engineered_df[cols_to_fill] = test_engineered_df.groupby('game_id')[cols_to_fill].ffill().fillna(0)

test_engineered_df['speed'].replace([np.inf, -np.inf], 0, inplace=True)

test_engineered_df['is_in_slot'] = (((test_engineered_df['shot_x'].between(X_FAR_LIMIT, X_NEAR_LIMIT)) & (test_engineered_df['shot_y'].abs() <= Y_LIMIT)) |
                    ((test_engineered_df['shot_x'].between(-X_NEAR_LIMIT, -X_FAR_LIMIT)) & (test_engineered_df['shot_y'].abs() <= Y_LIMIT))).astype(int)

test_engineered_df['shot_distance_binned'] = pd.cut(test_engineered_df['shot_distance'],
                                    bins=[0, 10, 25, 45, test_engineered_df['shot_distance'].max() + 1],
                                    labels=['Very_Close', 'Medium_Close', 'Medium_Far', 'Long_Shot'],
                                    right=False)

test_engineered_df['is_quick_release'] = (test_engineered_df['time_since_prev_event'] <= 3).astype(int)

test_engineered_df['is_prime_rebound'] = ((test_engineered_df['is_rebound'] == 1) &
                          (test_engineered_df['shot_distance'] <= 15) &
                          (test_engineered_df['time_since_prev_event'] <= 3)).astype(int)

test_engineered_df['speed_binned'] = pd.cut(test_engineered_df['speed'],
                            bins=[-1, 5, 10, 20, test_engineered_df['speed'].max() + 1],
                            labels=['Slow', 'Medium', 'Fast', 'Very_Fast'],
                            right=False)

test_engineered_df['angle_change'] = test_engineered_df['angle_change'].fillna(0)

drop_ids = ['player_id', 'hitter_id', 'hittee_id', 'shooter_id',
            'goalie_id', 'owner_team_id', 'home_id', 'away_id',
            'losing_player_id', 'winning_player_id', 'details.typeCode',
            'committed_by_id', 'drawn_by_id', 'served_by_id',
            'blocker_id', 'scorer_id', 'assist1_id', 'assist2_id',
            'attackingDirection', 'game_id']
test_engineered_df.drop(columns=drop_ids, inplace=True, errors='ignore')


ohe_cols = ['shot_type', 'zone_code', 'prev_event_type', 
            'shot_distance_binned', 'speed_binned']
test_engineered_df = pd.get_dummies(test_engineered_df, columns=ohe_cols, drop_first=True, dummy_na=False)

test_engineered_df['time_in_period_sec'] = test_engineered_df['time_in_period'].apply(time_to_seconds)
test_engineered_df['time_remaining_period_sec'] = test_engineered_df['time_remaining_period'].apply(time_to_seconds)
test_engineered_df.drop(columns=['time_in_period', 'time_remaining_period'], inplace=True)

scaling_cols = ['time_in_period_sec', 'time_remaining_period_sec', 
                'shot_distance', 'shot_angle', 'game_time_sec', 
                'prev_shot_distance', 'time_since_prev_event', 
                'dist_from_prev_event','angle_change', 'speed']

scaler = StandardScaler()
test_engineered_df[scaling_cols] = scaler.fit_transform(test_engineered_df[scaling_cols])

test_engineered_df = test_engineered_df[test_engineered_df['event_type_desc'].isin(EVENT_TYPES)]
test_engineered_df['is_goal'] = (test_engineered_df['event_type_desc'] == 'goal').astype(int)
test_engineered_df.drop(columns=['game_situation_code', 'event_type_desc'], inplace=True, errors='ignore')

bool_cols = test_engineered_df.select_dtypes(include='bool').columns
if len(bool_cols) > 0:
    test_engineered_df[bool_cols] = test_engineered_df[bool_cols].astype(int)

Performing additional engineering features to add to the features engineered in Part 4...


C:\Users\yosri\AppData\Local\Temp\ipykernel_36328\2567744817.py:87: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_engineered_df['speed'].replace([np.inf, -np.inf], 0, inplace=True)


In [62]:
test_engineered_df.to_csv("6_test_playoff_engineered_df.csv", index=False)

In [63]:
test_engineered_df.columns

Index(['period_number', 'max_reg_periods', 'away_sog', 'home_sog',
       'score_away', 'score_home', 'shot_x', 'shot_y', 'empty_net',
       'shot_distance', 'shot_angle', 'is_goal', 'periodDescriptor.otPeriods',
       'game_time_sec', 'time_since_prev_event', 'dist_from_prev_event',
       'is_rebound', 'prev_shot_distance', 'angle_change', 'speed',
       'is_empty_net', 'is_even_strength', 'is_power_play', 'is_shorthanded',
       'is_in_slot', 'is_quick_release', 'is_prime_rebound',
       'shot_type_deflected', 'shot_type_slap', 'shot_type_snap',
       'shot_type_tip-in', 'shot_type_wrap-around', 'shot_type_wrist',
       'zone_code_N', 'zone_code_O', 'prev_event_type_delayed-penalty',
       'prev_event_type_faceoff', 'prev_event_type_game-end',
       'prev_event_type_giveaway', 'prev_event_type_goal',
       'prev_event_type_hit', 'prev_event_type_missed-shot',
       'prev_event_type_penalty', 'prev_event_type_period-end',
       'prev_event_type_period-start', 'prev_event_